# 05 — Generate Initial Invoice History

## Purpose

This notebook generates a deterministic initial invoice history for the billing source system.

Each invoice is derived from a valid subscription billing cycle and includes pricing, discounts, usage overage, taxes, payment allocation, and lifecycle status.

The generated dataset will later support Bronze ingestion, payment reconciliation, revenue-leakage detection, customer-level financial analytics, and Gold-layer KPIs.

## Business Grain

One row represents one issued invoice for one subscription billing cycle.

## Invoice History

- Last 6 eligible billing cycles for monthly subscriptions
- Up to 2 eligible billing cycles for annual subscriptions
- Deterministic invoice identifiers
- Subscription-aligned billing periods
- Country-based synthetic tax rules
- Deterministic overage charges
- Paid, Open, Past Due, and Voided invoice states

## Data Quality Controls

- Required business-key validation
- Invoice identifier uniqueness
- Customer referential integrity
- Subscription referential integrity
- Subscription/customer relationship validation
- Invoice and subscription temporal consistency
- Billing-period validation
- Discount calculation reconciliation
- Contracted amount reconciliation
- Tax and invoice-total reconciliation
- Paid, outstanding, and voided balance reconciliation
- Controlled domain validation
- Post-write validation

## Inputs

- `/Volumes/workspace/revenue_leakage_bronze/landing/crm/customers/initial_load`
- `/Volumes/workspace/revenue_leakage_bronze/landing/subscription_system/subscriptions/initial_load`

## Target

- `/Volumes/workspace/revenue_leakage_bronze/landing/billing_system/invoices/initial_load`

## 1. Configuration and Schemas

Define deterministic generation parameters, source locations, the invoice target, and explicit schemas for all source and output datasets.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    IntegerType,
    DecimalType,
    BooleanType,
)

INITIAL_CUSTOMER_COUNT = 5000
INITIAL_SUBSCRIPTION_COUNT = 6000

MONTHLY_HISTORY_MONTHS = 6
ANNUAL_HISTORY_YEARS = 2

SNAPSHOT_DATE = "2026-08-15"

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/landing"
)

CUSTOMERS_INITIAL_PATH = (
    f"{LANDING_PATH}/crm/customers/initial_load"
)

SUBSCRIPTIONS_INITIAL_PATH = (
    f"{LANDING_PATH}/subscription_system/"
    "subscriptions/initial_load"
)

INVOICES_INITIAL_PATH = (
    f"{LANDING_PATH}/billing_system/invoices/initial_load"
)

CUSTOMER_REFERENCE_SCHEMA = StructType([
    StructField(
        "customer_id",
        StringType(),
        False
    ),
    StructField(
        "country",
        StringType(),
        False
    ),
])

SUBSCRIPTION_SCHEMA = StructType([
    StructField(
        "subscription_id",
        StringType(),
        False
    ),
    StructField(
        "customer_id",
        StringType(),
        False
    ),
    StructField(
        "plan_id",
        StringType(),
        False
    ),
    StructField(
        "plan_name",
        StringType(),
        False
    ),
    StructField(
        "start_date",
        DateType(),
        False
    ),
    StructField(
        "end_date",
        DateType(),
        True
    ),
    StructField(
        "subscription_status",
        StringType(),
        False
    ),
    StructField(
        "billing_frequency",
        StringType(),
        False
    ),
    StructField(
        "billing_day",
        IntegerType(),
        False
    ),
    StructField(
        "base_monthly_price",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "discount_percentage",
        DecimalType(5, 2),
        False
    ),
    StructField(
        "contracted_monthly_price",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "contracted_billing_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "discount_start_date",
        DateType(),
        True
    ),
    StructField(
        "discount_end_date",
        DateType(),
        True
    ),
    StructField(
        "included_usage_units",
        IntegerType(),
        False
    ),
    StructField(
        "overage_unit_price",
        DecimalType(10, 4),
        False
    ),
    StructField(
        "payment_terms_days",
        IntegerType(),
        False
    ),
    StructField(
        "auto_renew",
        BooleanType(),
        False
    ),
    StructField(
        "currency",
        StringType(),
        False
    ),
    StructField(
        "operation",
        StringType(),
        False
    ),
    StructField(
        "event_timestamp",
        TimestampType(),
        False
    ),
    StructField(
        "snapshot_date",
        DateType(),
        False
    ),
])

INVOICE_SCHEMA = StructType([
    StructField(
        "invoice_id",
        StringType(),
        False
    ),
    StructField(
        "subscription_id",
        StringType(),
        False
    ),
    StructField(
        "customer_id",
        StringType(),
        False
    ),
    StructField(
        "billing_period_start",
        DateType(),
        False
    ),
    StructField(
        "billing_period_end",
        DateType(),
        False
    ),
    StructField(
        "invoice_date",
        DateType(),
        False
    ),
    StructField(
        "due_date",
        DateType(),
        False
    ),
    StructField(
        "billing_frequency",
        StringType(),
        False
    ),
    StructField(
        "currency",
        StringType(),
        False
    ),
    StructField(
        "list_price_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "discount_percentage",
        DecimalType(5, 2),
        False
    ),
    StructField(
        "discount_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "net_subscription_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "overage_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "subtotal_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "tax_rate",
        DecimalType(5, 2),
        False
    ),
    StructField(
        "tax_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "invoice_total_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "amount_paid",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "outstanding_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "voided_amount",
        DecimalType(12, 2),
        False
    ),
    StructField(
        "invoice_status",
        StringType(),
        False
    ),
    StructField(
        "payment_terms_days",
        IntegerType(),
        False
    ),
    StructField(
        "operation",
        StringType(),
        False
    ),
    StructField(
        "event_timestamp",
        TimestampType(),
        False
    ),
    StructField(
        "snapshot_date",
        DateType(),
        False
    ),
])

INVOICE_COLUMNS = INVOICE_SCHEMA.fieldNames()

## 2. Load and Validate Source Snapshots

Load the initial customer and subscription snapshots using explicit schemas.

Before invoice generation begins, validate source row counts, business-key uniqueness, and customer referential integrity. The notebook fails immediately if an upstream source does not satisfy its data contract.

In [0]:
customers_reference_df = (
    spark.read
    .schema(CUSTOMER_REFERENCE_SCHEMA)
    .json(CUSTOMERS_INITIAL_PATH)
)

subscriptions_initial_df = (
    spark.read
    .schema(SUBSCRIPTION_SCHEMA)
    .json(SUBSCRIPTIONS_INITIAL_PATH)
)

customer_count = customers_reference_df.count()
subscription_count = subscriptions_initial_df.count()

duplicate_customer_key_count = (
    customers_reference_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_subscription_key_count = (
    subscriptions_initial_df
    .groupBy("subscription_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

orphan_subscription_customer_count = (
    subscriptions_initial_df
    .select("customer_id")
    .distinct()
    .join(
        customers_reference_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

assert customer_count == INITIAL_CUSTOMER_COUNT, (
    f"Expected {INITIAL_CUSTOMER_COUNT:,} customers, "
    f"but found {customer_count:,}."
)

assert subscription_count == INITIAL_SUBSCRIPTION_COUNT, (
    f"Expected {INITIAL_SUBSCRIPTION_COUNT:,} subscriptions, "
    f"but found {subscription_count:,}."
)

assert duplicate_customer_key_count == 0, (
    "Duplicate customer identifiers were detected."
)

assert duplicate_subscription_key_count == 0, (
    "Duplicate subscription identifiers were detected."
)

assert orphan_subscription_customer_count == 0, (
    "Subscriptions referencing unknown customers were detected."
)

print(f"Customer references loaded: {customer_count:,}")
print(f"Subscriptions loaded: {subscription_count:,}")
print(
    "Duplicate customer keys: "
    f"{duplicate_customer_key_count:,}"
)
print(
    "Duplicate subscription keys: "
    f"{duplicate_subscription_key_count:,}"
)
print(
    "Orphan subscription customers: "
    f"{orphan_subscription_customer_count:,}"
)

display(
    subscriptions_initial_df
    .groupBy(
        "billing_frequency",
        "subscription_status"
    )
    .count()
    .orderBy(
        "billing_frequency",
        "subscription_status"
    )
)

## 3. Build the Subscription Billing Schedule

Generate eligible invoice cycles from the subscription contracts.

Monthly subscriptions receive up to six recent billing cycles. Annual subscriptions receive up to two recent anniversary cycles. Billing dates are aligned with subscription start dates, configured billing days, and the snapshot boundary.

Invoice identifiers are generated deterministically from the subscription identifier and invoice date, avoiding global window operations and nondeterministic sequencing.

In [0]:
monthly_cycle_offsets_df = (
    spark.range(MONTHLY_HISTORY_MONTHS)
    .select(
        F.col("id")
        .cast("int")
        .alias("cycle_offset")
    )
)

monthly_invoice_schedule_df = (
    subscriptions_initial_df
    .filter(
        F.col("billing_frequency") == "Monthly"
    )
    .crossJoin(monthly_cycle_offsets_df)
    .withColumn(
        "cycle_month_start",
        F.expr(
            f"trunc("
            f"add_months("
            f"to_date('{SNAPSHOT_DATE}'), "
            f"-cycle_offset"
            f"), "
            f"'month'"
            f")"
        )
    )
    .withColumn(
        "billing_period_start",
        F.expr(
            "least("
            "date_add("
            "cycle_month_start, "
            "billing_day - 1"
            "), "
            "last_day(cycle_month_start)"
            ")"
        )
    )
    .withColumn(
        "next_cycle_month_start",
        F.add_months(
            F.col("cycle_month_start"),
            1
        )
    )
    .withColumn(
        "next_billing_date",
        F.expr(
            "least("
            "date_add("
            "next_cycle_month_start, "
            "billing_day - 1"
            "), "
            "last_day(next_cycle_month_start)"
            ")"
        )
    )
    .withColumn(
        "billing_period_end",
        F.date_sub(
            F.col("next_billing_date"),
            1
        )
    )
    .withColumn(
        "invoice_date",
        F.col("billing_period_start")
    )
    .filter(
        (
            F.col("invoice_date")
            >= F.col("start_date")
        )
        & (
            F.col("invoice_date")
            <= F.lit(SNAPSHOT_DATE).cast("date")
        )
    )
)

annual_invoice_schedule_df = (
    subscriptions_initial_df
    .filter(
        F.col("billing_frequency") == "Annual"
    )
    .withColumn(
        "completed_annual_cycles",
        F.floor(
            F.months_between(
                F.lit(SNAPSHOT_DATE).cast("date"),
                F.col("start_date")
            ) / 12
        ).cast("int")
    )
    .withColumn(
        "first_annual_cycle",
        F.greatest(
            F.col("completed_annual_cycles")
            - F.lit(ANNUAL_HISTORY_YEARS - 1),
            F.lit(0)
        )
    )
    .withColumn(
        "annual_cycle_index",
        F.explode(
            F.sequence(
                F.col("first_annual_cycle"),
                F.col("completed_annual_cycles")
            )
        )
    )
    .withColumn(
        "invoice_date",
        F.expr(
            "add_months("
            "start_date, "
            "annual_cycle_index * 12"
            ")"
        )
    )
    .withColumn(
        "billing_period_start",
        F.col("invoice_date")
    )
    .withColumn(
        "billing_period_end",
        F.date_sub(
            F.add_months(
                F.col("invoice_date"),
                12
            ),
            1
        )
    )
    .filter(
        F.col("invoice_date")
        <= F.lit(SNAPSHOT_DATE).cast("date")
    )
)

SCHEDULE_COLUMNS = [
    "subscription_id",
    "customer_id",
    "start_date",
    "billing_period_start",
    "billing_period_end",
    "invoice_date",
    "billing_frequency",
    "base_monthly_price",
    "discount_percentage",
    "contracted_billing_amount",
    "overage_unit_price",
    "payment_terms_days",
    "currency",
]

invoice_schedule_df = (
    monthly_invoice_schedule_df
    .select(*SCHEDULE_COLUMNS)
    .unionByName(
        annual_invoice_schedule_df
        .select(*SCHEDULE_COLUMNS)
    )
    .withColumn(
        "invoice_id",
        F.concat(
            F.lit("INV-"),
            F.col("subscription_id"),
            F.lit("-"),
            F.date_format(
                F.col("invoice_date"),
                "yyyyMMdd"
            )
        )
    )
    .withColumn(
        "due_date",
        F.expr(
            "date_add("
            "invoice_date, "
            "payment_terms_days"
            ")"
        )
    )
)

monthly_invoice_count = (
    monthly_invoice_schedule_df.count()
)

annual_invoice_count = (
    annual_invoice_schedule_df.count()
)

scheduled_invoice_count = (
    invoice_schedule_df.count()
)

distinct_scheduled_invoice_count = (
    invoice_schedule_df
    .select("invoice_id")
    .distinct()
    .count()
)

assert scheduled_invoice_count == (
    monthly_invoice_count + annual_invoice_count
), "Invoice schedule row counts do not reconcile."

assert distinct_scheduled_invoice_count == (
    scheduled_invoice_count
), "Duplicate scheduled invoice identifiers were detected."

assert scheduled_invoice_count > 0, (
    "The generated invoice schedule is empty."
)

print(
    f"Monthly invoice cycles: "
    f"{monthly_invoice_count:,}"
)

print(
    f"Annual invoice cycles: "
    f"{annual_invoice_count:,}"
)

print(
    f"Scheduled invoices: "
    f"{scheduled_invoice_count:,}"
)

print(
    f"Distinct scheduled invoice IDs: "
    f"{distinct_scheduled_invoice_count:,}"
)

display(
    invoice_schedule_df
    .groupBy("billing_frequency")
    .count()
    .orderBy("billing_frequency")
)

## 4. Calculate Invoice Financials and Lifecycle Status

Enrich each scheduled invoice with deterministic financial calculations and lifecycle attributes.

The calculation applies list pricing, contractual discounts, synthetic usage-overage charges, country-based tax rates, and invoice totals. Each invoice is then assigned a deterministic Paid, Open, Past Due, or Voided status.

Paid, outstanding, and voided amounts are stored separately to support downstream payment reconciliation and revenue-leakage analysis.

In [0]:
invoice_calculation_df = (
    invoice_schedule_df
    .join(
        customers_reference_df,
        on="customer_id",
        how="inner"
    )
    .withColumn(
        "list_price_amount",
        F.when(
            F.col("billing_frequency") == "Annual",
            F.col("base_monthly_price") * 12
        )
        .otherwise(
            F.col("base_monthly_price")
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "discount_amount",
        F.round(
            F.col("list_price_amount")
            * F.col("discount_percentage")
            / 100,
            2
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "net_subscription_amount",
        (
            F.col("list_price_amount")
            - F.col("discount_amount")
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "overage_bucket",
        F.pmod(
            F.xxhash64("invoice_id"),
            F.lit(100)
        )
    )
    .withColumn(
        "overage_amount",
        F.when(
            F.col("overage_bucket") < 25,
            F.round(
                (
                    F.pmod(
                        F.xxhash64(
                            F.concat(
                                "invoice_id",
                                F.lit("-usage")
                            )
                        ),
                        F.lit(2500)
                    )
                    + F.lit(100)
                )
                * F.col("overage_unit_price"),
                2
            )
        )
        .otherwise(
            F.lit(0.00)
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "subtotal_amount",
        (
            F.col("net_subscription_amount")
            + F.col("overage_amount")
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "tax_rate",
        F.when(
            F.col("country") == "United States",
            0.00
        )
        .when(
            F.col("country") == "Canada",
            5.00
        )
        .when(
            F.col("country") == "Germany",
            19.00
        )
        .when(
            F.col("country") == "France",
            20.00
        )
        .when(
            F.col("country") == "United Kingdom",
            20.00
        )
        .when(
            F.col("country") == "Netherlands",
            21.00
        )
        .otherwise(0.00)
        .cast(DecimalType(5, 2))
    )
    .withColumn(
        "tax_amount",
        F.round(
            F.col("subtotal_amount")
            * F.col("tax_rate")
            / 100,
            2
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "invoice_total_amount",
        (
            F.col("subtotal_amount")
            + F.col("tax_amount")
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "status_bucket",
        F.pmod(
            F.xxhash64(
                F.concat(
                    "invoice_id",
                    F.lit("-status")
                )
            ),
            F.lit(100)
        )
    )
    .withColumn(
        "invoice_status",
        F.when(
            F.col("status_bucket") < 82,
            "Paid"
        )
        .when(
            F.col("status_bucket") >= 98,
            "Voided"
        )
        .when(
            F.col("due_date")
            >= F.lit(SNAPSHOT_DATE).cast("date"),
            "Open"
        )
        .otherwise("Past Due")
    )
    .withColumn(
        "amount_paid",
        F.when(
            F.col("invoice_status") == "Paid",
            F.col("invoice_total_amount")
        )
        .otherwise(
            F.lit(0.00)
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "outstanding_amount",
        F.when(
            F.col("invoice_status").isin(
                "Open",
                "Past Due"
            ),
            F.col("invoice_total_amount")
        )
        .otherwise(
            F.lit(0.00)
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "voided_amount",
        F.when(
            F.col("invoice_status") == "Voided",
            F.col("invoice_total_amount")
        )
        .otherwise(
            F.lit(0.00)
        )
        .cast(DecimalType(12, 2))
    )
    .withColumn(
        "operation",
        F.lit("INSERT")
    )
    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.concat(
                F.date_format(
                    F.col("invoice_date"),
                    "yyyy-MM-dd"
                ),
                F.lit(" 08:00:00")
            )
        )
    )
    .withColumn(
        "snapshot_date",
        F.lit(SNAPSHOT_DATE).cast("date")
    )
)

invoices_initial_df = (
    invoice_calculation_df
    .select(*INVOICE_COLUMNS)
)

generated_invoice_count = (
    invoices_initial_df.count()
)

assert generated_invoice_count == scheduled_invoice_count, (
    "Generated invoice count does not match "
    "the billing schedule."
)

print(
    f"Generated invoices: "
    f"{generated_invoice_count:,}"
)

display(
    invoices_initial_df
    .groupBy(
        "billing_frequency",
        "invoice_status"
    )
    .count()
    .orderBy(
        "billing_frequency",
        "invoice_status"
    )
)

display(
    invoices_initial_df
    .orderBy(
        F.col("invoice_date").desc(),
        "invoice_id"
    )
    .limit(20)
)

## 5. Validate Invoice Data Quality and Financial Reconciliation

Apply fail-fast validation across structural, referential, temporal, financial, and business-rule dimensions.

The controls verify invoice uniqueness, required fields, valid customer and subscription relationships, billing chronology, contractual pricing, tax calculations, invoice totals, lifecycle status, and the reconciliation of paid, outstanding, and voided balances.

In [0]:
actual_invoice_count = (
    invoices_initial_df.count()
)

distinct_invoice_count = (
    invoices_initial_df
    .select("invoice_id")
    .distinct()
    .count()
)

null_required_condition = F.lit(False)

for column_name in INVOICE_COLUMNS:
    null_required_condition = (
        null_required_condition
        | F.col(column_name).isNull()
    )

null_required_field_count = (
    invoices_initial_df
    .filter(null_required_condition)
    .count()
)

null_business_key_count = (
    invoices_initial_df
    .filter(
        F.col("invoice_id").isNull()
        | F.col("subscription_id").isNull()
        | F.col("customer_id").isNull()
        | F.col("event_timestamp").isNull()
    )
    .count()
)

duplicate_event_count = (
    invoices_initial_df
    .groupBy(
        "invoice_id",
        "operation",
        "event_timestamp"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

orphan_subscription_count = (
    invoices_initial_df
    .select("subscription_id")
    .distinct()
    .join(
        subscriptions_initial_df
        .select("subscription_id"),
        on="subscription_id",
        how="left_anti"
    )
    .count()
)

orphan_customer_count = (
    invoices_initial_df
    .select("customer_id")
    .distinct()
    .join(
        customers_reference_df
        .select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

subscription_customer_mismatch_count = (
    invoices_initial_df
    .alias("invoice")
    .join(
        subscriptions_initial_df
        .select(
            "subscription_id",
            "customer_id"
        )
        .alias("subscription"),
        on=(
            F.col("invoice.subscription_id")
            == F.col("subscription.subscription_id")
        ),
        how="inner"
    )
    .filter(
        F.col("invoice.customer_id")
        != F.col("subscription.customer_id")
    )
    .count()
)

invoice_before_subscription_count = (
    invoices_initial_df
    .alias("invoice")
    .join(
        subscriptions_initial_df
        .select(
            "subscription_id",
            "start_date"
        )
        .alias("subscription"),
        on=(
            F.col("invoice.subscription_id")
            == F.col("subscription.subscription_id")
        ),
        how="inner"
    )
    .filter(
        F.col("invoice.invoice_date")
        < F.col("subscription.start_date")
    )
    .count()
)

invoice_after_snapshot_count = (
    invoices_initial_df
    .filter(
        F.col("invoice_date")
        > F.col("snapshot_date")
    )
    .count()
)

invalid_billing_period_count = (
    invoices_initial_df
    .filter(
        (
            F.col("billing_period_start")
            > F.col("billing_period_end")
        )
        | (
            F.col("invoice_date")
            != F.col("billing_period_start")
        )
        | (
            F.col("due_date")
            < F.col("invoice_date")
        )
    )
    .count()
)

discount_mismatch_count = (
    invoices_initial_df
    .filter(
        F.abs(
            F.col("discount_amount")
            - F.round(
                F.col("list_price_amount")
                * F.col("discount_percentage")
                / 100,
                2
            )
        ) > 0.01
    )
    .count()
)

net_amount_mismatch_count = (
    invoices_initial_df
    .filter(
        F.abs(
            F.col("net_subscription_amount")
            - (
                F.col("list_price_amount")
                - F.col("discount_amount")
            )
        ) > 0.01
    )
    .count()
)

contracted_amount_mismatch_count = (
    invoices_initial_df
    .alias("invoice")
    .join(
        invoice_schedule_df
        .select(
            "invoice_id",
            "contracted_billing_amount"
        )
        .alias("schedule"),
        on="invoice_id",
        how="inner"
    )
    .filter(
        F.abs(
            F.col("invoice.net_subscription_amount")
            - F.col(
                "schedule.contracted_billing_amount"
            )
        ) > 0.01
    )
    .count()
)

subtotal_mismatch_count = (
    invoices_initial_df
    .filter(
        F.abs(
            F.col("subtotal_amount")
            - (
                F.col("net_subscription_amount")
                + F.col("overage_amount")
            )
        ) > 0.01
    )
    .count()
)

tax_mismatch_count = (
    invoices_initial_df
    .filter(
        F.abs(
            F.col("tax_amount")
            - F.round(
                F.col("subtotal_amount")
                * F.col("tax_rate")
                / 100,
                2
            )
        ) > 0.01
    )
    .count()
)

invoice_total_mismatch_count = (
    invoices_initial_df
    .filter(
        F.abs(
            F.col("invoice_total_amount")
            - (
                F.col("subtotal_amount")
                + F.col("tax_amount")
            )
        ) > 0.01
    )
    .count()
)

balance_reconciliation_error_count = (
    invoices_initial_df
    .filter(
        F.abs(
            F.col("invoice_total_amount")
            - (
                F.col("amount_paid")
                + F.col("outstanding_amount")
                + F.col("voided_amount")
            )
        ) > 0.01
    )
    .count()
)

invalid_status_allocation_count = (
    invoices_initial_df
    .filter(
        (
            (F.col("invoice_status") == "Paid")
            & (
                (
                    F.col("amount_paid")
                    != F.col("invoice_total_amount")
                )
                | (
                    F.col("outstanding_amount")
                    != 0
                )
                | (
                    F.col("voided_amount")
                    != 0
                )
            )
        )
        | (
            F.col("invoice_status").isin(
                "Open",
                "Past Due"
            )
            & (
                (
                    F.col("outstanding_amount")
                    != F.col("invoice_total_amount")
                )
                | (
                    F.col("amount_paid")
                    != 0
                )
                | (
                    F.col("voided_amount")
                    != 0
                )
            )
        )
        | (
            (F.col("invoice_status") == "Voided")
            & (
                (
                    F.col("voided_amount")
                    != F.col("invoice_total_amount")
                )
                | (
                    F.col("amount_paid")
                    != 0
                )
                | (
                    F.col("outstanding_amount")
                    != 0
                )
            )
        )
    )
    .count()
)

invalid_status_date_count = (
    invoices_initial_df
    .filter(
        (
            (F.col("invoice_status") == "Open")
            & (
                F.col("due_date")
                < F.col("snapshot_date")
            )
        )
        | (
            (F.col("invoice_status") == "Past Due")
            & (
                F.col("due_date")
                >= F.col("snapshot_date")
            )
        )
    )
    .count()
)

invalid_domain_count = (
    invoices_initial_df
    .filter(
        ~F.col("invoice_status").isin(
            "Paid",
            "Open",
            "Past Due",
            "Voided"
        )
        | ~F.col("billing_frequency").isin(
            "Monthly",
            "Annual"
        )
        | (
            F.col("currency") != "USD"
        )
        | (
            F.col("operation") != "INSERT"
        )
        | (
            F.col("discount_percentage") < 0
        )
        | (
            F.col("discount_percentage") > 100
        )
        | (
            F.col("tax_rate") < 0
        )
        | (
            F.col("tax_rate") > 25
        )
        | (
            F.col("list_price_amount") < 0
        )
        | (
            F.col("discount_amount") < 0
        )
        | (
            F.col("net_subscription_amount") < 0
        )
        | (
            F.col("overage_amount") < 0
        )
        | (
            F.col("tax_amount") < 0
        )
        | (
            F.col("invoice_total_amount") < 0
        )
        | (
            F.col("amount_paid") < 0
        )
        | (
            F.col("outstanding_amount") < 0
        )
        | (
            F.col("voided_amount") < 0
        )
    )
    .count()
)

assert actual_invoice_count == scheduled_invoice_count, (
    "Invoice count does not match the billing schedule."
)

assert distinct_invoice_count == actual_invoice_count, (
    "Duplicate invoice identifiers were detected."
)

assert null_required_field_count == 0, (
    "Null values were detected in required invoice fields."
)

assert null_business_key_count == 0, (
    "Null invoice business keys were detected."
)

assert duplicate_event_count == 0, (
    "Duplicate invoice events were detected."
)

assert orphan_subscription_count == 0, (
    "Invoices referencing unknown subscriptions were detected."
)

assert orphan_customer_count == 0, (
    "Invoices referencing unknown customers were detected."
)

assert subscription_customer_mismatch_count == 0, (
    "Invoice customer and subscription customer do not match."
)

assert invoice_before_subscription_count == 0, (
    "Invoices issued before subscription start were detected."
)

assert invoice_after_snapshot_count == 0, (
    "Invoices issued after the snapshot date were detected."
)

assert invalid_billing_period_count == 0, (
    "Invalid invoice billing periods were detected."
)

assert discount_mismatch_count == 0, (
    "Invoice discount calculation errors were detected."
)

assert net_amount_mismatch_count == 0, (
    "Net subscription amount errors were detected."
)

assert contracted_amount_mismatch_count == 0, (
    "Invoice amounts do not match subscription contracts."
)

assert subtotal_mismatch_count == 0, (
    "Invoice subtotal errors were detected."
)

assert tax_mismatch_count == 0, (
    "Invoice tax calculation errors were detected."
)

assert invoice_total_mismatch_count == 0, (
    "Invoice total calculation errors were detected."
)

assert balance_reconciliation_error_count == 0, (
    "Invoice balance reconciliation errors were detected."
)

assert invalid_status_allocation_count == 0, (
    "Invalid invoice status balance allocations were detected."
)

assert invalid_status_date_count == 0, (
    "Invoice status and due-date inconsistencies were detected."
)

assert invalid_domain_count == 0, (
    "Invalid invoice domain values were detected."
)

print(
    f"Validated invoice rows: "
    f"{actual_invoice_count:,}"
)

print(
    f"Distinct invoice IDs: "
    f"{distinct_invoice_count:,}"
)

print(
    f"Null required fields: "
    f"{null_required_field_count:,}"
)

print(
    f"Null business keys: "
    f"{null_business_key_count:,}"
)

print(
    f"Duplicate events: "
    f"{duplicate_event_count:,}"
)

print(
    f"Orphan subscriptions: "
    f"{orphan_subscription_count:,}"
)

print(
    f"Orphan customers: "
    f"{orphan_customer_count:,}"
)

print(
    f"Subscription/customer mismatches: "
    f"{subscription_customer_mismatch_count:,}"
)

print(
    f"Invoices before subscription start: "
    f"{invoice_before_subscription_count:,}"
)

print(
    f"Invoices after snapshot: "
    f"{invoice_after_snapshot_count:,}"
)

print(
    f"Invalid billing periods: "
    f"{invalid_billing_period_count:,}"
)

print(
    f"Discount mismatches: "
    f"{discount_mismatch_count:,}"
)

print(
    f"Net amount mismatches: "
    f"{net_amount_mismatch_count:,}"
)

print(
    f"Contracted amount mismatches: "
    f"{contracted_amount_mismatch_count:,}"
)

print(
    f"Subtotal mismatches: "
    f"{subtotal_mismatch_count:,}"
)

print(
    f"Tax mismatches: "
    f"{tax_mismatch_count:,}"
)

print(
    f"Invoice total mismatches: "
    f"{invoice_total_mismatch_count:,}"
)

print(
    f"Balance reconciliation errors: "
    f"{balance_reconciliation_error_count:,}"
)

print(
    f"Invalid status allocations: "
    f"{invalid_status_allocation_count:,}"
)

print(
    f"Invalid status dates: "
    f"{invalid_status_date_count:,}"
)

print(
    f"Invalid domain values: "
    f"{invalid_domain_count:,}"
)

display(
    invoices_initial_df
    .groupBy("invoice_status")
    .agg(
        F.count("*").alias("invoice_count"),
        F.round(
            F.sum("invoice_total_amount"),
            2
        ).alias("invoiced_amount"),
        F.round(
            F.sum("amount_paid"),
            2
        ).alias("paid_amount"),
        F.round(
            F.sum("outstanding_amount"),
            2
        ).alias("outstanding_amount"),
        F.round(
            F.sum("voided_amount"),
            2
        ).alias("voided_amount")
    )
    .orderBy("invoice_status")
)

## 6. Persist and Revalidate the Raw Invoice History

Persist the validated invoice history as raw JSON in the billing-system landing directory.

The dataset is reloaded using the explicit invoice schema and validated again for row-count preservation, identifier uniqueness, required fields, lifecycle distributions, and financial-total preservation.

In [0]:
# The synthetic initial invoice history is fully
# regenerated on every run.
(
    invoices_initial_df.write
    .format("json")
    .mode("overwrite")
    .save(INVOICES_INITIAL_PATH)
)

saved_invoices_df = (
    spark.read
    .schema(INVOICE_SCHEMA)
    .json(INVOICES_INITIAL_PATH)
)

saved_invoice_count = (
    saved_invoices_df.count()
)

saved_distinct_invoice_count = (
    saved_invoices_df
    .select("invoice_id")
    .distinct()
    .count()
)

saved_null_required_condition = F.lit(False)

for column_name in INVOICE_COLUMNS:
    saved_null_required_condition = (
        saved_null_required_condition
        | F.col(column_name).isNull()
    )

saved_null_required_field_count = (
    saved_invoices_df
    .filter(saved_null_required_condition)
    .count()
)

source_status_counts_df = (
    invoices_initial_df
    .groupBy("invoice_status")
    .count()
)

saved_status_counts_df = (
    saved_invoices_df
    .groupBy("invoice_status")
    .count()
)

status_count_mismatch_count = (
    source_status_counts_df
    .exceptAll(saved_status_counts_df)
    .count()
    + saved_status_counts_df
    .exceptAll(source_status_counts_df)
    .count()
)

source_financial_summary = (
    invoices_initial_df
    .agg(
        F.round(
            F.sum("invoice_total_amount"),
            2
        ).alias("invoice_total_amount"),
        F.round(
            F.sum("amount_paid"),
            2
        ).alias("amount_paid"),
        F.round(
            F.sum("outstanding_amount"),
            2
        ).alias("outstanding_amount"),
        F.round(
            F.sum("voided_amount"),
            2
        ).alias("voided_amount")
    )
    .first()
)

saved_financial_summary = (
    saved_invoices_df
    .agg(
        F.round(
            F.sum("invoice_total_amount"),
            2
        ).alias("invoice_total_amount"),
        F.round(
            F.sum("amount_paid"),
            2
        ).alias("amount_paid"),
        F.round(
            F.sum("outstanding_amount"),
            2
        ).alias("outstanding_amount"),
        F.round(
            F.sum("voided_amount"),
            2
        ).alias("voided_amount")
    )
    .first()
)

assert saved_invoice_count == actual_invoice_count, (
    "Saved invoice row count does not match "
    "the generated dataset."
)

assert saved_distinct_invoice_count == saved_invoice_count, (
    "Duplicate invoice identifiers were found "
    "after persistence."
)

assert saved_null_required_field_count == 0, (
    "Null required fields were found "
    "after persistence."
)

assert status_count_mismatch_count == 0, (
    "Invoice status distributions changed "
    "during persistence."
)

assert (
    saved_financial_summary["invoice_total_amount"]
    == source_financial_summary["invoice_total_amount"]
), "Invoice totals changed during persistence."

assert (
    saved_financial_summary["amount_paid"]
    == source_financial_summary["amount_paid"]
), "Paid totals changed during persistence."

assert (
    saved_financial_summary["outstanding_amount"]
    == source_financial_summary["outstanding_amount"]
), "Outstanding totals changed during persistence."

assert (
    saved_financial_summary["voided_amount"]
    == source_financial_summary["voided_amount"]
), "Voided totals changed during persistence."

print(
    f"Saved invoice rows: "
    f"{saved_invoice_count:,}"
)

print(
    f"Saved distinct invoice IDs: "
    f"{saved_distinct_invoice_count:,}"
)

print(
    f"Saved null required fields: "
    f"{saved_null_required_field_count:,}"
)

print(
    f"Status count mismatches: "
    f"{status_count_mismatch_count:,}"
)

print(
    f"Saved invoice total: "
    f"{saved_financial_summary['invoice_total_amount']}"
)

print(
    f"Target path: "
    f"{INVOICES_INITIAL_PATH}"
)

display(
    saved_invoices_df
    .groupBy(
        "billing_frequency",
        "invoice_status"
    )
    .count()
    .orderBy(
        "billing_frequency",
        "invoice_status"
    )
)